# CodeBuddy — Fine-tuning Gemma 4 dengan Unsloth

Notebook ini melakukan fine-tuning Gemma 4 menggunakan dataset tutoring Bahasa Indonesia CodeBuddy.

**Runtime yang dibutuhkan:** GPU T4 (gratis di Colab) atau lebih baik

**Estimasi waktu:** 4–8 jam untuk full training

**Target:** $10,000 Unsloth Prize di Google Gemma 4 Good Hackathon

In [ ]:
# Step 1: Install Unsloth
!pip install unsloth -q
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo -q

In [ ]:
# Step 2: Load Gemma 4 dengan Unsloth
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-4-e2b-it-unsloth-bnb-4bit',  # E2B fit di T4
    max_seq_length=2048,
    load_in_4bit=True,
)

print('Model loaded:', model.config.model_type)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 3: Tambahkan LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=42,
)

In [ ]:
# Step 4: Load dan convert dataset CodeBuddy
import json
from datasets import Dataset

# Upload file ini ke Colab atau load dari Google Drive
# File: backend/data/finetune_dataset.json

with open('finetune_dataset.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

def convert_to_conversations(item):
    """Convert format CodeBuddy ke format chat Unsloth."""
    return {
        'conversations': [
            {
                'role': 'user',
                'content': f"{item['instruction']}\n\n{item['input']}"
            },
            {
                'role': 'assistant', 
                'content': item['output']
            }
        ]
    }

converted = [convert_to_conversations(item) for item in raw['data']]
dataset = Dataset.from_list(converted)

print(f'Dataset: {len(dataset)} contoh')
print('Contoh pertama:')
print(json.dumps(dataset[0], indent=2, ensure_ascii=False)[:500])

In [ ]:
# Step 5: Apply chat template
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')
dataset = standardize_sharegpt(dataset)

def format_prompts(examples):
    texts = tokenizer.apply_chat_template(
        examples['conversations'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': texts}

dataset = dataset.map(format_prompts, batched=True)
print('Dataset siap. Sample:')
print(dataset[0]['text'][:300])

In [ ]:
# Step 6: Training
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,  # 3 epoch cukup untuk 55 contoh
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='codebuddy-gemma4-checkpoints',
        report_to='none',
    ),
)

print('Mulai training...')
trainer_stats = trainer.train()
print(f'Training selesai! Loss akhir: {trainer_stats.training_loss:.4f}')

In [ ]:
# Step 7: Test model hasil fine-tuning
from unsloth.chat_templates import get_chat_template

FastModel.for_inference(model)

test_input = [
    {
        'role': 'user',
        'content': 'Analisis kode Python ini dan jelaskan errornya dalam Bahasa Indonesia\n\nprin("Halo Dunia")'
    }
]

inputs = tokenizer.apply_chat_template(
    test_input,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('Respons model fine-tuned:')
print(response)

In [ ]:
# Step 8: Simpan model ke format GGUF (untuk Ollama)
model.save_pretrained_gguf(
    'codebuddy-gemma4-finetuned',
    tokenizer,
    quantization_method='q4_k_m',  # Balance kualitas vs ukuran
)
print('Model GGUF tersimpan!')
print('Upload file .gguf ke Ollama dengan:')
print('  ollama create codebuddy -f Modelfile')

In [ ]:
# Step 9: Upload ke Hugging Face Hub (untuk submission hackathon)
# Ganti dengan username HF kamu
HF_USERNAME = 'username-kamu'
HF_TOKEN = 'hf_...'  # dari https://huggingface.co/settings/tokens

model.push_to_hub_gguf(
    f'{HF_USERNAME}/codebuddy-gemma4-indonesian-tutor',
    tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN,
)
print(f'Model tersedia di: https://huggingface.co/{HF_USERNAME}/codebuddy-gemma4-indonesian-tutor')

## Setelah Fine-tuning Selesai

### Gunakan model di Ollama (lokal)

```bash
# Buat Modelfile
cat > Modelfile << 'EOF'
FROM ./codebuddy-gemma4-finetuned/unsloth.Q4_K_M.gguf
SYSTEM "Kamu adalah CodeBuddy, tutor Python untuk pelajar Indonesia. Selalu jawab dalam Bahasa Indonesia yang ramah dan menyemangati."
EOF

# Buat model Ollama
ollama create codebuddy -f Modelfile

# Test
ollama run codebuddy "Jelaskan apa itu variabel dalam Python"
```

### Update config CodeBuddy
```python
# utils/config.py
ollama_model: str = "codebuddy"  # pakai model fine-tuned
```